# Deep Learning Competition 1 
112078502 楊婷婷  
[2025 DataLab Cup1 : Predicting News Popularity](https://www.kaggle.com/competitions/2025-data-lab-cup-1-predicting-news-popularity/leaderboard)  


Report  
1. How did you preprocess data, e.g. cleaning, feature engineering, etc?  
2. How did you build the classifier, e.g. model, training algorithm, special techniques, etc?
3. Conclusions, including interesting findings, pitfalls, takeaway lessons, etc.


In [21]:
# 0. import models
# models
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score, train_test_split
# pre-process
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk
from bs4 import BeautifulSoup
import re
# regular
import pandas as pd
import numpy as np
from pandarallel import pandarallel
import pdb
from IPython.display import display

In [22]:
# 1. Load data
df_train = pd.read_csv('./dataset/train.csv')
df_test = pd.read_csv('./dataset/test.csv') 

## 1. How did you preprocess data, e.g. cleaning, feature engineering, etc? 

#### 1-1. Extract and cleaning
Use BeautifulSoup library to extract infomation using html tags, and use Regular expression operations(re) to remove all prefix or tailing whitespace from the text

#### 1-2. Feature engineering : choose and process the text information
<style>
li {
  margin-bottom: 0.6em;  /* add space between bullet points */
  line-height: 1.5em;    /* make each line inside a bullet more readable */
}
</style>
* Choose **"Author"** : split into multiple author -> replace blank with '_' to combine the first and last name of single author -> use '&' to separate different authors -> put all authors into single string for a column e.g "by John Smith, Jane Doe & Bob Lee" becomes "john_smith jane_doe bob_lee"
* Besides **"Author"**, I also choose **"Title"**, **"Channel"** (Category of the articles, e.g tech, entertainment...), **'Topic'** (article labels, multiple categories tags)

#### 1-3. Feature engineering : choose and process the number information
* Existing information: map string type of **"Day"**, **"Month"** to a number
* Get rid of "Min" and "Second", assuming it won't affect the popularity. Keep **"Hour"**, e.g during commuting hours may lead to more clicks  
* Choose "Content_Len", "Num_See_Also", "Num_Image", "Num_A" (number of hyperlinks)
* <font color="red"><b>Self-defined New fatures :</b></font>
    * <font color="red"><b>"Timestamp"</b></font> : As the Month, Day, Year are stored in separated columns, it cannot reflect the timeline infomation !   
    It preserve true time spacing, and can help to capture true chronological distance between samples, sequential trends, combined seasonality patterns — full yearly or weekly cycles, not just per-field cycles.  
    -> This significantly improve the performance from <span style="color:red">0.58923 to 0.59212</span> in public case, and <span style="color:red">0.58595 to 0.59054</span> in private case
    * "Month_sin", "Month_cos" to capture cyclic information (e.g let Dec close to Jan), to detect seasonality information
    -> it slightly improve the performance


In [23]:
# --------------------------------------------
# 2-1 Extract features
# --------------------------------------------

def extract_features(html_text):
    soup = BeautifulSoup(html_text, 'html.parser')

    # Title 
    title_tag = soup.find('h1', class_='title')
    title = title_tag.get_text(strip=True).lower() if title_tag else ''

    # Author
    info_div = soup.find('div', class_='article-info')
    author = ''
    if info_div:
        name_tag = info_div.find('span', class_='author_name')
        if name_tag:
            author = name_tag.get_text(strip=True)
        else:
            span = info_div.find('span')
            author = span.get_text(strip=True) if span else (
                info_div.a.get_text(strip=True) if info_div.a else '')
    author = re.sub(r'\s+', ' ', author.strip().lower())
    author = re.sub(r'^by\s+', '', author)
    author = author.replace(' and ', ' & ')
    author = re.sub(r'&.*;', '&', author)

    # handle multiple authors
    if ',' in author:
        parts = [p.strip() for p in re.split(r'\s*,\s*', author)]
        if '&' in parts[-1]:
            more = [p.strip() for p in re.split(r'\s*&\s*', parts[-1])]
            parts = parts[:-1] + more
        author_list = parts
    else:
        author_list = [a.strip() for a in re.split(r'\s*&\s*', author)]
    author = ' '.join([re.sub(r'\s+', '_', a) for a in author_list])

    # Channel 
    channel = ''
    article_tag = soup.find('article')
    if article_tag and article_tag.has_attr('data-channel'):
        channel = article_tag['data-channel'].strip().lower()

    # Topics 
    topic_footer = soup.find('footer', class_='article-topics')
    topic_labels = ''
    if topic_footer:
        topics = [a.get_text(strip=True).lower().replace(' ', '_')
                  for a in topic_footer.find_all('a')]
        topics = list(dict.fromkeys(topics))
        topic_labels = ' '.join(topics)

    # Datetime
    month_map = {'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
                 'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12}
    day_map = {'mon': 1, 'tue': 2, 'wed': 3, 'thu': 4,
               'fri': 5, 'sat': 6, 'sun': 7}
    day_weight_map = {1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.23, 6: 1.23, 7: 1.23}

    date_time = ''
    if info_div and info_div.find('time'):
        date_time = info_div.find('time').get('datetime', '')
    else:
        date_time = 'Wed, 10 Oct 2014 15:00:43'

    match = re.search(
        r'([\w]+),\s+(\d+)\s+([\w]+)\s+(\d+)\s+(\d+):(\d+):(\d+)', date_time)
    if match:
        day_str, date, month_str, year, hour, minute, second = match.groups()
        day = day_map.get(day_str.lower(), 0)
        month = month_map.get(month_str.lower(), 0)
    else:
        day = month = date = year = hour = minute = second = 0
    
    day_weight = day_weight_map.get(day, 0)

    # Content
    content_section = soup.find('section', class_='article-content')
    content_text = content_section.get_text(
        separator=' ', strip=True) if content_section else ''
    content_len = len(content_text)
    num_links = len(content_section.find_all('a')) if content_section else 0
    num_images = len(content_section.find_all('img')) if content_section else 0
    num_see_also = len(re.findall(r'see also', content_text.lower()))

    return (title, author, channel, topic_labels, day, day_weight, int(date), int(month), int(year),
            int(hour), content_len, num_see_also, num_images, num_links)

# --------------------------------------------
# 2-2 Apply parallel to speed up
# --------------------------------------------
pandarallel.initialize(progress_bar=False, verbose=0)
train_features = df_train['Page content'].parallel_apply(extract_features)
test_features = df_test['Page content'].parallel_apply(extract_features)

# --------------------------------------------
# 2-3 create df and Adding new customized columns
# --------------------------------------------
df_features = pd.DataFrame(
    list(train_features) + list(test_features),
    columns=[
        'Title', 'Author', 'Channel', 'Topic', 'Day' ,'Day_Weight', 'Date', 'Month', 'Year', 'Hour', 'Content_Len', 'Num_See_Also', 'Num_Image', 'Num_A'
    ]
)
df_features['Timestamp'] = pd.to_datetime({
    'year': df_features['Year'],
    'month': df_features['Month'],
    'day': df_features['Date']  # 'Date' is actually the day of month
}, errors='coerce')
df_features['Timestamp'] = df_features['Timestamp'].apply(
    lambda x: x.toordinal() if pd.notna(x) else np.nan
)

df_features['Month_sin'] = np.sin(2 * np.pi * df_features['Month'] / 12)
df_features['Month_cos'] = np.cos(2 * np.pi * df_features['Month'] / 12)

# --------------------------------------------
# 2-4 Define text_cols, and num_cols, and Cleaning NaN data
# --------------------------------------------
text_cols = ['Title', 'Author', 'Channel', 'Topic']
num_cols = ['Day', 'Day_Weight', 'Date', 'Month', 'Year', 'Hour', 'Content_Len', 'Num_See_Also', 'Num_Image', 'Num_A', 'Timestamp','Month_sin', 'Month_cos']

df_features[text_cols] = df_features[text_cols].fillna('').astype(str)
df_features[num_cols] = df_features[num_cols].fillna(0).astype(float)

# show head
df_features.head()

,Title,Author,Channel,Topic,Day,Day_Weight,Date,Month,Year,Hour,Content_Len,Num_See_Also,Num_Image,Num_A,Timestamp,Month_sin,Month_cos
0,nasa's grand challenge: stop asteroids from de...,clara_moskowitz,world,asteroid asteroids challenge earth space u.s. ...,3.0,1.00,19.0,6.0,2013.0,15.0,3600.0,4.0,0.0,14.0,735038.0,1.224647e-16,-1.000000e+00
1,google's new open source patent pledge: we won...,bychristina_warren,tech,apps_and_software google open_source opn_pledg...,4.0,1.00,28.0,3.0,2013.0,17.0,1842.0,1.0,0.0,8.0,734955.0,1.000000e+00,6.123234e-17
2,ballin': 2014 nfl draft picks get to choose th...,bysam_laird,entertainment,entertainment nfl nfl_draft sports television,3.0,1.00,7.0,5.0,2014.0,19.0,6399.0,1.0,0.0,4.0,735360.0,5.000000e-01,-8.660254e-01
3,cameraperson fails deliver slapstick laughs,bysam_laird,watercooler,sports video videos watercooler,5.0,1.23,11.0,10.0,2013.0,2.0,1624.0,1.0,0.0,7.0,735152.0,-8.660254e-01,5.000000e-01
4,nfl star helps young fan prove friendship with...,byconnor_finnegan,entertainment,entertainment instagram instagram_video nfl sp...,4.0,1.00,17.0,4.0,2014.0,3.0,8351.0,1.0,50.0,9.0,735340.0,8.660254e-01,-5.000000e-01


<span style="font-size:32px">training and testing</span>  

Above features process all columns in both the train.csv and test.cv. Split the first of df_training rows to training data(with ground truth's popularity labels ) , and rest for testing data 

In [24]:
# --------------------------------------------
# 3. Training and Testing data
# --------------------------------------------
# training data
X_all = df_features.iloc[:len(df_train), :].copy()
y_all = (df_train['Popularity'].values == 1).astype(int)
# testing data
X_test = df_features.iloc[len(df_train):, :].copy()

#### 1-4. Pre-process text data
<style>
li {
  margin-bottom: 0.6em;  /* add space between bullet points */
  line-height: 1.5em;    /* make each line inside a bullet more readable */
}
</style>
* tokeinizer and lemmatizer   
  After each text combine into single string, we need to split and tokenize them.  
  * Tokenize: 
  (1) # e.g "We'll" keep the word before '-> We 
  (2) remove char besided a-z0-9, '_' (as we use them to combine author and topics)
  * Lemmatize: get the stem of the word, e.g  books ->book (noun), stopped -> stop (verb). I don't apply adj and adv's 
  * filter out stop word

* Column Transform
We can define different preprocessing pipelines for different columns  
-> text_cols using CountVectorizer  
-> num_cols using Standarization  
Finally, then merges all of them into one unified feature matrix  

* <font color="red">I have also tried with TfidfVectorizer </font> rather than CountVectorizer, but the result is worse! 

In [25]:
# --------------------------------------------
# 4-1. Tokenizer & lemmatizer
# --------------------------------------------
import random
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('stopwords', quiet=True)
wnl = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def tokenize_title(text: str): # tokenize + lemmatize
    text = text.lower()
    text = re.sub(r"([\w]+)'[\w]+", r"\1", text)  # e.g "We'll" keep the word before '-> We
    text = re.sub(r'[^a-z0-9_]+', ' ', text)    # remove any char besides a-z 0-9, '_' for author
    tokens = [tok for tok in text.split() if tok] # tokenize
    lemmas = []
    for tok in tokens:
        lemma = wnl.lemmatize(tok, pos='n')   # lemmatize: noun base
        lemma = wnl.lemmatize(lemma, pos='v') # lemmatize: verb base
        if lemma not in stop_words:           # filter out stop_words
            lemmas.append(lemma)

    return lemmas

# ------------------------
# 4-2. Column transformer
# ------------------------
def make_text_pipe():  # make single pipe
    return Pipeline([
        ('cv', CountVectorizer(tokenizer=tokenize_title, lowercase=False, token_pattern=None))
        # ('tfidf', TfidfVectorizer(tokenizer=tokenize_title, lowercase=False, token_pattern=None)) # replace the cv
        # ('to_dense', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)) # convert sparse to dense matrix (if some models requires it)
    ])
# def make_text_pipe_TFIDF():  # this is worse then countVectorizer
#     return Pipeline([
#         ('tfidf', TfidfVectorizer(tokenizer=tokenize_title, lowercase=False, token_pattern=None)) # replace the cv
#     ])

# output data type: scipy.sparse.csr_matrix

# to get the  — usually a sparse matrix (using ColumnTransformer)
trans_all = ColumnTransformer(
    transformers=[
        ('title_text',   make_text_pipe(), 'Title'),
        ('author_text',  make_text_pipe(), 'Author'),
        ('channel_text', make_text_pipe(), 'Channel'),
        ('topic_text',   make_text_pipe(), 'Topic'),
        ('num', StandardScaler(), num_cols)
    ],
    # sparse_threshold=1.0,   # default threshold=0.3, set to 1.0 if want it to be sparse
    remainder='drop',
    n_jobs=-1
)



In [26]:
# some print check

# print check before and after tokenization
sample = X_all.sample(5, random_state=0)
sample['Title_tokens']  = sample['Title'].apply(tokenize_title)
sample['Author_tokens'] = sample['Author'].apply(tokenize_title)
sample['Topic_tokens']  = sample['Topic'].apply(tokenize_title)
print("=== Before and After Tokenization ===")
print(sample[['Title', 'Title_tokens']])
# print(sample[['Title', 'Title_tokens', 'Author', 'Author_tokens', 'Topic', 'Topic_tokens']])

print("\n--- title afte tokenize ---")
idx = random.choice(df_features.index)
sample_df = df_features.loc[[idx]]  # get as df
print("Random Title:", sample_df['Title'].iloc[0])
cv = CountVectorizer(tokenizer=tokenize_title, lowercase=False, token_pattern=None)
sample_title_vec = cv.fit_transform([sample_df['Title'].iloc[0]])  # wrap in list!

print("\n--- CountVectorizer ---")
print("Feature names:", cv.get_feature_names_out())
print(sample_title_vec)
print("Type:", type(sample_title_vec))
print("Shape:", sample_title_vec.shape)
print("\n--- ColumnTransformer ---")
sample_all_vec = trans_all.fit_transform(sample_df)
print("Type:", type(sample_all_vec))
print("Shape:", sample_all_vec.shape)
print(sample_all_vec)

=== Before and After Tokenization ===
                                                   Title  \
13516      14 digital tools for authors and illustrators   
11607  money talks: who's afraid of facebook's whatsa...   
11866                 next-generation xbox coming may 21   
25540  the fix to facebook blackouts is sitting right...   
26611              8 tech terms explained by non-techies   

                                          Title_tokens  
13516         [14, digital, tool, author, illustrator]  
11607  [money, talk, afraid, facebook, whatsapp, deal]  
11866          [next, generation, xbox, come, may, 21]  
25540     [fix, facebook, blackout, sit, right, front]  
26611            [8, tech, term, explain, non, techie]  

--- title afte tokenize ---
Random Title: katy perry's victorian dreams go up in flames in 'unconditionally'

--- CountVectorizer ---
Feature names: ['dream' 'flame' 'go' 'katy' 'perry' 'unconditionally' 'victorian']
<Compressed Sparse Row sparse matrix of d

<style>
li {
  margin-bottom: 0.6em;  /* add space between bullet points */
  line-height: 1.0em;    /* make each line inside a bullet more readable */
}
</style>
## 2. How did you build the classifier, e.g. model, training algorithm, special techniques, etc?
* Define the parameter setting for each models first and then using **Pipeline** as a processing sequence blueprint that instructs what order it will execute (not execute yet), (ColumnTransformer-> Apply model parameter)
* Note: In scikit-learn, a **Pipeline** is a chain (or sequence) of data-processing steps executed in order.


#### 2-1. Choosing Model and Evaluation

tree models(lgbm, catboost) capture interactions, linear models(sgd) capture global structure

For none-linear model : 
* **LGBM** :  
Handles non-linear feature interactions with gradient-based tree splits. It can learn complex boundaries, robust to outliers, and provide strong AUC with minimal tuning.  

* **CatBoost**:   
It similar to LGBM, but better regularization categorical handling. It can complement bias vs LGBM, often improves ensemble stability.  

For linear model : 
* **SGD**:  
captures global linear trends, to habe interpretable weights, and serves as a low-bias, high-variance complement to tree-based models.

#### 2-2. Tuning the parameter (using LGBM for example)
* n_estimator: number of trees, Each tree in the ensemble learns to correct the errors of the previous trees.
    If Higher te n_estimators:
    - Allows the model to learn more complex patterns
    - May lead to overfitting (models becomes too complex)
    Lower is otherwise.
* Interaction with n_estimator and learning_ rate :
    - A lower learning_rate (e.g., 0.0001) requires a higher n_estimators to achieve good performance.
    - A higher te learning rate (e.g., 0.1) can work with a lower n_estimators 

#### 2-3 Combining multiple models
* <font color="red">Why single model always has the best performance, but we still have to combine multiple models?4</font>

All tree models can make similar mistakes on certain feature combinations. A linear model or another boosting variant might err differently. By combining them, you average out uncorrelated errors, which increases the AUC stability across folds or unseen data.  
In plain words, even if LGBM is “strong,” it still has its blind spots — it may overfit certain patterns or ignore global linear trends. Other models cover those weaknesses.

* The method we use the combining the models?

We use <font color="red">stacking</font> rather than voting. Voting is a simple weighted average. Conversely, stacking is a meta-model (often a logistic regression or another booster) that learns the optimal combination of base models, so it can find nonlinear boundaries between model outputs that best separate classes, to improve AUC.

#### 2-4 other experiments
refer to:  
<font color="red">5-1: use scoring to find the feature that is important</font>  
<font color="red">5-2: auto tuning the paramaters for lgbm </font>

In [27]:
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

lgbm = LGBMClassifier(
    random_state=0,
    learning_rate=0.0045, # 0.009
    n_estimators=400, # 300
    verbose=-1,
)

cat = CatBoostClassifier(
    n_estimators=400, # 300
    random_state=0,
    verbose=0,
)

sgd = SGDClassifier(
    loss='log_loss',      # logistic regression (probabilities)
    penalty='l2',         # or 'elasticnet' for L1+L2 regularization
    alpha=1e-4,           # regularization strength
    max_iter=1000,
    tol=1e-3,
    random_state=0,
    n_jobs=-1
)
# pipeline
pipe_lgbm = Pipeline([
    ('ct', trans_all), 
    ('clf', lgbm)
])
pipe_cat = Pipeline([
    ('ct', trans_all),
    ('clf', cat)
])
pipe_sgd = Pipeline([
    ('ct', trans_all),  
    ('clf', sgd)
])

from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import RidgeClassifier

stacking = StackingClassifier(
    estimators=[
        ('lgbm', pipe_lgbm),
        ('cat', pipe_cat),
        ('sgd', pipe_sgd)
    ],
    final_estimator=LogisticRegression(
        max_iter=500,
        solver='lbfgs'
    ),
    stack_method='predict_proba',  # use predicted probabilities from base models
    n_jobs=1, #change to avoid parallel running conflicts
    cv=5 # cv='prefit' if need to speed up
)


models = {
    "LightGBM": pipe_lgbm,
    "CatBoost": pipe_cat,
    "SGD" : pipe_sgd,
    "Stacking": stacking
}


print("=== Cross-validation AUC per fold ===")
for name, model in models.items():
    scores = cross_val_score(model, X_all, y_all, cv=3, scoring='roc_auc', n_jobs=1)
    print(f"{name:12s}: {scores}  ->  mean={scores.mean():.5f}, std={scores.std():.5f}")

=== Cross-validation AUC per fold ===
LightGBM    : [0.6032062  0.60291518 0.60966946]  ->  mean=0.60526, std=0.00312
CatBoost    : [0.5957884  0.59113017 0.59708781]  ->  mean=0.59467, std=0.00256
SGD         : [0.5461461  0.54641177 0.54099521]  ->  mean=0.54452, std=0.00249
Stacking    : [0.60461892 0.602381   0.60803295]  ->  mean=0.60501, std=0.00232


#### 2-5. Fit chosen model and Output files

In [28]:
chosen_model = stacking
chosen_model.fit(X_all, y_all)

y_score = chosen_model.predict_proba(X_test)[:, 1]
y_score = np.round(y_score, 1) # round to 1 decimal

df_pred = pd.DataFrame({
    'Id': df_test['Id'],
    'Popularity': y_score
})
df_pred.to_csv('submission.csv', index=False, float_format='%.1f')
print(df_pred.head(10))


      Id  Popularity
0  27643         0.4
1  27644         0.4
2  27645         0.4
3  27646         0.7
4  27647         0.5
5  27648         0.4
6  27649         0.7
7  27650         0.7
8  27651         0.8
9  27652         0.5


## 3. Conclusions, including interesting findings, pitfalls, takeaway lessons, etc.
<style>
li {
  margin-bottom: 0.6em;  /* add space between bullet points */
  line-height: 1.5em;    /* make each line inside a bullet more readable */
}
</style>

1. Feature affects a lot ! Adding the timestamp does have stable improvement, as states in 1-1 
2. Coninue with the feature extraction, after writing the report, I modify the small error, which is tokenize is removing the '_', e.g authr = "Neil_Young" becomes ["Neil", "Young"] which is wrong... After fixing it, the private score improves a lot ! from <font color="red">0.59098 to 0.59209</font>, but it's too late
3. I extract the feature to see which one is more important ! I found the "tiimestamp" I created does matter ! (refer to 5-1) 
4. I have tried to auto tuning the paramter for lgbm model and as two of final submission, but the result is getting worse (the part of code to tune refer to 6-2)
5. Maybe I should have used more model for stacking to increase generalizability

## 4. Final Result
| Private | Public | Description |
|----------|---------|-------------|
| 0.59098  | 0.59761 | Stacking (3 models) + manual tuning of LightGBM parameters |
| 0.58684  | 0.59117 | Stacking (2 models, excluding SGD) + auto tuning of LightGBM parameters |


## 5. Other experiements
####  5-1 LightGBM's find the feature importance

In [32]:
# Train the pipeline (including lgbm model)
pipe_lgbm.fit(X_all, y_all)
# Extract the trained LightGBM model from the pipeline
trained_lgbm = pipe_lgbm.named_steps['clf']
# Get feature importances
feature_importances = trained_lgbm.feature_importances_
# Extract actual feature names from the ColumnTransformer
feature_names = pipe_lgbm.named_steps['ct'].get_feature_names_out()
# Map feature importances to feature names
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
})
# final print
importance_df = importance_df.sort_values(by='Importance', ascending=False)
print(importance_df)

                      Feature  Importance
30287          num__Timestamp        1983
30277                num__Day         874
30282               num__Hour         712
30280              num__Month         568
30283        num__Content_Len         533
...                       ...         ...
10135     title_text__persist           0
10134     title_text__persian           0
10133  title_text__persephone           0
10132     title_text__perseid           0
10146    title_text__persuade           0

[30290 rows x 2 columns]


#### 5-2 LightGBM's early-stopping automatically finds the best hyperparameter of boosting rounds

In [30]:
# *** Continue from the same code of 1-1 to 1-4 , and the rest of 2-1-2-3 changed

from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import uniform, randint
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

print("=== Running RandomizedSearchCV for LightGBM (80K dataset) ===")
# Split training set for early stopping
X_train, X_valid, y_train, y_valid = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

# Pre-transform once (no pipeline inside the search)
X_train_t = trans_all.fit_transform(X_train)
X_valid_t = trans_all.transform(X_valid)

# Define base model to be trained for parameter finding 
lgbm_base = LGBMClassifier(
    random_state=0,
    n_estimators=3000, # large so that can find untill valid score is decreasing
    verbose=-1 
)

# Parameter distributions
from scipy.stats import loguniform
param_dist = {
    'learning_rate': loguniform(0.001, 0.1),
    'num_leaves': randint(15, 255),
    'max_depth': randint(3, 12),
    'min_child_samples': randint(10, 200),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4)
}

# Fit parameters for early stopping
fit_params = {
    'eval_set': [(X_valid_t, y_valid)],
    'callbacks': [
        early_stopping(stopping_rounds=100, verbose=False),
        # log_evaluation(period=50)
    ]
}

# Randomized search on numeric matrix (not pipeline)
search = RandomizedSearchCV(
    estimator=lgbm_base,
    param_distributions=param_dist,
    n_iter=20,
    scoring='roc_auc',
    cv=3, # to be quicker
    random_state=42,
    verbose=0,
    n_jobs=1
)
search.fit(X_train_t, y_train, **fit_params)

# print searched result 
print("Best params:", search.best_params_)
print(f"Best AUC: {search.best_score_:.5f}")
best_iter = search.best_estimator_.best_iteration_
print(f"Best iteration: {best_iter}")
best_params = search.best_params_


# model and pipeline
final_lgbm = LGBMClassifier(
    random_state=0,
    learning_rate=best_params['learning_rate'],
    num_leaves=best_params['num_leaves'],
    max_depth=best_params['max_depth'],
    min_child_samples=best_params['min_child_samples'],
    subsample=best_params['subsample'],
    colsample_bytree=best_params['colsample_bytree'],
    n_estimators=best_iter,
    eval_metric='auc',
    verbose=-1
)

# pipeline
pipe_lgbm_trained = Pipeline([
    ('ct', trans_all), 
    ('clf', final_lgbm) # change to final_lgbm
])

models = {
    "LightGBM": pipe_lgbm_trained,
}

print("=== Cross-validation AUC per fold ===")
for name, model in models.items():
    scores = cross_val_score(model, X_all, y_all, cv=3, scoring='roc_auc', n_jobs=1)
    print(f"{name:12s}: {scores}  ->  mean={scores.mean():.5f}, std={scores.std():.5f}")

=== Running RandomizedSearchCV for LightGBM (80K dataset) ===


Best params: {'colsample_bytree': np.float64(0.662397808134481), 'learning_rate': np.float64(0.0013066739238053278), 'max_depth': 10, 'min_child_samples': 126, 'num_leaves': 114, 'subsample': np.float64(0.6571467271687763)}
Best AUC: 0.60340
Best iteration: 1965
=== Cross-validation AUC per fold ===
LightGBM    : [0.60284794 0.60383083 0.60503462]  ->  mean=0.60390, std=0.00089
